# Quiz 04: Web APIs and JSON

This quiz covers web APIs and JSON, from lectures 18 and 19.

- You have 75 minutes. The quiz is open-book and open-notes
- Fork and clone this repository, and work in your clone
- The data files you need are already in the `data/` folder
- Run the cells from top to bottom, and run them all again before you submit
- Tasks 4, 6, 7, 8, 9 and 10 send real requests to live APIs
- Tasks 7 and 10 run `curl` in the terminal. Paste the commands and their output into the answer cells

Write your answers in the empty cells below each task. The bonus task is optional.


In [2]:
import json

import requests
import pandas as pd

## Task 1

Look at this URL:

```
https://api.worldbank.org/v2/country/URY/indicator/SP.POP.TOTL?format=json
```

Write the scheme, the host, the path and the query string, one on each line. Then write the full URL for France's population from 2015 to 2018.

**Your answer:**


**Scheme:** `https`  
**Host:** `api.worldbank.org`  
**Path:** `/v2/country/URY/indicator/SP.POP.TOTL`  
**Query string:** `format=json`

**France's population, 2015–2018:**

```
https://api.worldbank.org/v2/country/FRA/indicator/SP.POP.TOTL?format=json&date=2015:2018
```

## Task 2

Load `data/shuttle.json` with `json.load` into a variable called `reply`. Print:

1. the name of the stop
2. how many arrivals the reply lists
3. the wait for the second arrival, with its unit, for example `5 minutes`


In [3]:
# Task 2
with open("data/shuttle.json") as f:
    reply = json.load(f)

print(reply["stop"]["name"])
print(len(reply["arrivals"]))
print(reply["arrivals"][1]["wait"], reply["units"]["wait"])


Emory Point
3
11 minutes


## Task 3

Load `data/wb_electricity_ken.json` into a variable called `wb`. It holds the share of Kenya's (KEN) population with access to electricity. Like many APIs, the World Bank sends a list with two elements: metadata first, then the records. Print the number of records the metadata says there are, and check it against the length of the records list. Then print the value for 2020.


In [9]:
# Task 3

with open("data/wb_electricity_ken.json") as f:
    wb = json.load(f)

meta = wb[0]
records = wb[1]
year_2020 = records[4]

print(meta["total"])
print(len(records))
print(year_2020["date"], year_2020["value"])


13
13
2020 71.5


## Task 4

Ask [Open-Meteo](https://open-meteo.com/en/docs) (`https://api.open-meteo.com/v1/forecast`) for the current wind speed at Emory: latitude 33.79, longitude -84.32, and `wind_speed_10m` as the `current` variable in the query. Pass the three values as a dict to `params`, with `timeout=10`, and save the reply as `r`. Print `r.url` and `r.status_code`. Then print the wind speed with its unit.


In [13]:
# Task 4

url = ("https://api.open-meteo.com/"
       "v1/forecast")
params = {
    "latitude": 33.79,
    "longitude": -84.32,
    "current": "wind_speed_10m",
}

r = requests.get(url, params=params, timeout=10)
print(r.url)
print(r.status_code)

data = r.json()
print(data["current"]["wind_speed_10m"], data["current_units"]["wind_speed_10m"])


https://api.open-meteo.com/v1/forecast?latitude=33.79&longitude=-84.32&current=wind_speed_10m
200
7.6 km/h


## Task 5

Without running any code, write the full URL that this call sends. Some characters must be percent-encoded in a URL. Encode them as `requests` would (lecture 18).

```python
requests.get("https://api.example.com/v2/books",
             params={"author": "Toni Morrison", "year": "1970:1990", "format": "json"})
```


**Your answer:**

https://api.example.com/v2/books?author=Toni+Morrison&year=1970%3A1990&format=json



## Task 6

Make two bad requests and print the status code and the body (`r.json()`) of each:

1. The Task 4 request with latitude 9999
2. `https://api.worldbank.org/v2/country/KEN/indicator/EG.ELC.ACCS.ZZ?format=json` (the indicator is misspelled)

For each, say in one sentence whether `raise_for_status()` would stop the script. Then write one line of code that would catch the second error.


**Your answer:**

In [20]:

#1
url = ("https://api.open-meteo.com/"
       "v1/forecast")
params = {
    "latitude": 9999,
    "longitude": -84.32,
    "current": "wind_speed_10m",
}

r1 = requests.get(url, params=params, timeout=10)
print(r1.status_code)
print(r1.json())


#Yes, `raise_for_status()` would stop the script, because Open-Meteo returns status 400, and any 4xx or 5xx status raises an error.


url2 = ("https://api.worldbank.org/v2/"
        "country/KEN/indicator/EG.ELC.ACCS.ZZ")
params2 = {
    "format": "json",
}

r2 = requests.get(url2, params=params2, timeout=10)
print(r2.status_code)
print(r2.json())

#2. No, `raise_for_status()` would not stop the script, because the World Bank returns status 200 even though the body only contains an error message instead of data.

# A line that would catch the  error
if len(r2.json()) < 2: print("No data:", r2.json())

400
{'reason': 'Latitude must be in range of -90 to 90°. Given: 9999.0.', 'error': True}
200
[{'message': [{'id': '120', 'key': 'Invalid value', 'value': 'The provided parameter value is not valid'}]}]
No data: [{'message': [{'id': '120', 'key': 'Invalid value', 'value': 'The provided parameter value is not valid'}]}]


## Task 7

In the terminal, use `curl` to ask [Open-Meteo](https://open-meteo.com/en/docs) (`https://api.open-meteo.com/v1/forecast`) for the weather in Montevideo: latitude -34.90, longitude -56.19, and `relative_humidity_2m` as the `current` variable. Paste into the answer cell, as text:

1. The `curl` command that prints only the status line, and its output
2. The `curl` command that saves the reply as `montevideo.json`, in the top folder of the repository
3. The Python command that prints the saved file indented, and the humidity with its unit


**Your answer:**

**Commands:**
 BASE="https://api.open-meteo.com/v1/forecast"
 Q="latitude=-34.90&longitude=-56.19&current=relative_humidity_2m"
 curl -si "$BASE?$Q" | head -n 1
 curl -s -o montevideo.json "$BASE?$Q"
 python3 -m json.tool montevideo.json


**Output:**
HTTP/1.1 200 OK
{
    "latitude": -34.90334,
    "longitude": -56.1929,
    "generationtime_ms": 0.03147125244140625,
    "utc_offset_seconds": 0,
    "timezone": "GMT",
    "timezone_abbreviation": "GMT",
    "elevation": 7.0,
    "current_units": {
        "time": "iso8601",
        "interval": "seconds",
        "relative_humidity_2m": "%"
    },
    "current": {
        "time": "2026-11-19T23:30",
        "interval": 900,
        "relative_humidity_2m": 82
    }
}






## Task 8

The script `scripts/pull_data.py` downloads one World Bank indicator and saves two files in `data/raw/`. Change its four settings to collect the share of the population using the internet in Kenya, Ghana and Senegal, from 2015 to 2024, with `internet` as the output name. Find the indicator code at [data.worldbank.org/indicator](https://data.worldbank.org/indicator).

Run the script from the top folder of this repository, and paste the settings you used and the script's full output into the answer cell. Then answer briefly:

1. How many rows did you get, and why does the script save the raw reply as well as the CSV?
2. Look at Ghana's values in `data/raw/internet.csv`. Would you trust the value for 2019? Say why, and what you would do about it in an analysis.


**Your answer:**
#AI assisted with the terminal issues, as the new script was not saving. Thus fully giving credit for the following command
sed -i '' \
  -e 's/^INDICATOR = .*/INDICATOR = "IT.NET.USER.ZS"/' \
  -e 's/^COUNTRIES = .*/COUNTRIES = "KEN;GHA;SEN"/' \
  -e 's/^YEARS = .*/YEARS = "2015:2024"/' \
  -e 's/^OUTPUT_NAME = .*/OUTPUT_NAME = "internet"/' \
  scripts/pull_data.py


In [22]:
INDICATOR = "IT.NET.USER.ZS"
COUNTRIES = "KEN;GHA;SEN"
YEARS = "2015:2024"
OUTPUT_NAME = "internet"


**Output:**

```
Requesting https://api.worldbank.org/v2/country/KEN;GHA;SEN/indicator/IT.NET.USER.ZS
Saved 30 rows to data/raw/internet.csv
  country_code country  year      value
0          GHA   Ghana  2015  23.000000
1          GHA   Ghana  2016  28.000000
2          GHA   Ghana  2017  37.884201
3          GHA   Ghana  2018  43.000000
4          GHA   Ghana  2019  22.073200
```

1. It returned back a tota of 30 rows: 3 countries × 10 years, with no missing values. The script
   saves the raw reply (`internet_raw.json`) as well as the CSV because the CSV
   is already processe. Thus, it keeps only four fields and drops missing values. The
   raw file keeps exactly what the API sent, so I can check the cleaning, redo it
   differently, or work offline without downloading the data again.

2. No I would not trust it, as Ghana's value falls from 43% in 2018 to 22.07% in 2019, then jumps to
   62.52% in 2020. Internet use doesn't half in one year and then nearly triple
   the next, so 2019 is very likely a data error. In an analysis I would flag it,
   check it against the original ITU source, and either treat it as missingor show results with/ without it.

## Task 9

[Frankfurter](https://frankfurter.dev/v1/) is a free API for exchange rates. Its latest rates are at `https://api.frankfurter.dev/v1/latest`, and it takes two options: `base`, the currency you convert from, and `symbols`, the currencies you want, separated by commas.

Using `requests` and `params`, ask for the latest rates from `USD` to `BRL` and `EUR`. Print `r.url`, the date of the rates, and the BRL rate.

For a range of dates, replace `latest` with `start..end`, for example `2020-01-06..2020-01-19` (two weeks, Monday to Sunday). Request that range with the same `params`. Print how many days of rates you got and the BRL rate on 15 January 2020. In one sentence, say why there are fewer than 14 days.


In [ ]:
# Task 9

url = ("https://api.frankfurter.dev/"
       "v1/latest")
params = {
    "base": "USD",
    "symbols": "BRL,EUR",
}

r = requests.get(url, params=params, timeout=10)
print(r.url)

data = r.json()
print(data["date"])
print(data["rates"]["BRL"])

# Two weeks in January 2020
url2 = ("https://api.frankfurter.dev/"
        "v1/2020-01-06..2020-01-19")

r2 = requests.get(url2, params=params, timeout=10)
data2 = r2.json()

print(len(data2["rates"]))
print(data2["rates"]["2020-01-15"]["BRL"])

https://api.frankfurter.dev/v1/latest?base=USD&symbols=BRL%2CEUR
2026-11-19
5.892


## Task 10

Frankfurter's two options are `base` and `symbols` (Task 9). Run these three commands, paste each reply, and say in one sentence how the API treated the options in each. Then say whether checking the status code alone would have caught any of the problems.

```bash
curl -s "https://api.frankfurter.dev/v1/latest?Base=USD&symbols=BRL"
curl -s "https://api.frankfurter.dev/v1/latest?base=USD&symbol=BRL"
curl -s "https://api.frankfurter.dev/v1/latest?base=usd&symbols=brl"
```


**Your answer:**

**Commands**

```
curl -s "https://api.frankfurter.dev/v1/latest?Base=USD&symbols=BRL"
{"amount":1.0,"base":"EUR","date":"2026-11-19","rates":{"BRL":6.4765}}
```

The API ignored `Base` because option names are case-sensitive, so it used its default base, EUR, and returned the EUR to BRL rate (6.4765) instead of USD to BRL.


```
curl -s "https://api.frankfurter.dev/v1/latest?base=USD&symbol=BRL"
{"amount":1.0,"base":"USD","date":"2026-11-19","rates":{"AUD":1.49,"BRL":5.892,"CAD":1.43,"CHF":0.84261,"CNY":6.6667,"CZK":22.223,"DKK":6.7997,"EUR":0.90975,"GBP":0.76038,"HKD":7.8456,"HUF":336.09,"IDR":18261,"ILS":3.2887,"INR":93.3,"ISK":124.82,"JPY":159.97,"KRW":1328.45,"MXN":18.9183,"MYR":4.1125,"NOK":9.6866,"NZD":1.8095,"PHP":63.695,"PLN":4.0184,"RON":4.8647,"SEK":10.141,"SGD":1.2897,"THB":33.905,"TRY":50.468,"ZAR":17.2806}}
```

The API used `base=USD` correctly but ignored the misspelled `symbol`, so it returned rates for all 29 currencies instead of only BRL.


```
curl -s "https://api.frankfurter.dev/v1/latest?base=usd&symbols=brl"
{"amount":1.0,"base":"USD","date":"2026-11-19","rates":{"BRL":5.892}}
```

The API accepted the lowercase currency codes and returned the correct USD to BRL rate (5.892), so the values are not case-sensitive, only the option names are.

**Status lines**

```
curl -si "https://api.frankfurter.dev/v1/latest?Base=USD&symbols=BRL" | head -n 1
HTTP/2 200
curl -si "https://api.frankfurter.dev/v1/latest?base=USD&symbol=BRL" | head -n 1
HTTP/2 200
curl -si "https://api.frankfurter.dev/v1/latest?base=usd&symbols=brl" | head -n 1
HTTP/2 200
```

**Response** No, since all three requests returned status 200, so the status code would not have caught either problem. The API silently ignored the wrong option names instead of returning an error, so you have to check the data itself, for example that `base` is USD and that `rates` only contains BRL.




## Bonus task

Write a `while` loop that reads `data/wb_gdppc_2023_page{n}.json`, starting at `n = 1`, and extends a list called `records`, which starts empty. Stop using the `pages` field from the metadata. Do not hard-code 3. Print `len(records)` at the end: it should match the `total` field, which is 265.

In [ ]:
# AI, specifically Claude was utilized to assist with error on questions, delete and extra folder created, and assist with clearing unclear terms in the questions.

When you are done, run all the cells from the top one more time and save the notebook. Then stage, commit and push everything to your fork: `quiz04.ipynb`, `montevideo.json`, your edited `scripts/pull_data.py` and the two files in `data/raw/`. Post your fork's link on Canvas.

Best of luck!